# Protótipo do modelo de classificação (Treinamento do Zero)

In [ ]:
import sys
import os

sys.path.append(os.path.abspath('../..'))

In [ ]:
# 1 - Importações
import tensorflow as tf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import random

In [ ]:
# 2 - Criação do modelo base (sem ImageNet)
base_model = tf.keras.applications.ResNet50(weights=None, include_top=False, input_shape=(224, 224, 3))

In [ ]:
# 3 - Camadas densas personalizadas
x = base_model.output
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(512, activation='relu')(x)
x = tf.keras.layers.Dense(256, activation='relu')(x)
x = tf.keras.layers.Dense(128, activation='relu')(x)
predis = tf.keras.layers.Dense(7, activation='softmax')(x)

modelo_classificacao = tf.keras.Model(inputs=base_model.input, outputs=predis)

In [ ]:
# 4 - Preparação dos dados
train_datagen = tf.keras.preprocessing.image.ImageDataGenerator(preprocessing_function=tf.keras.applications.resnet50.preprocess_input,
                                                                validation_split=0.2)

train_generator = train_datagen.flow_from_directory('./train_images_classification',
                                                    target_size=(224, 224),
                                                    color_mode='rgb',
                                                    batch_size=32,
                                                    class_mode='categorical',
                                                    shuffle=True,
                                                    subset='training')

In [ ]:
# 5 - Compilação e Treinamento do Modelo
modelo_classificacao.compile(optimizer='Adam', loss='categorical_crossentropy', metrics=['accuracy'])

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

history = modelo_classificacao.fit(train_generator, epochs=40)

In [ ]:
# 6 - Salvando o modelo
from utils.models_pkl import save_model
save_model(modelo_classificacao, "modelo_classificacao_do_zero")

In [ ]:
# 7 - Avaliação do Modelo
accuracy = history.history["accuracy"]
loss = history.history["loss"]

plt.figure()
plt.plot(accuracy, label="Evolução da Acurácia Durante Treinamento")
plt.xlabel("Epochs")
plt.ylabel("Acurácia")
plt.legend()

plt.figure()
plt.plot(loss, label="Evolução da Perda Durante Treinamento")
plt.xlabel("Epochs")
plt.ylabel("Perda")
plt.legend()